# Imports, Installations and Downloads

In [11]:
import torch as t
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split, DataLoader
from torchvision import datasets, transforms
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

# Data Processing

This notebook uses the datasets and transforms modules from the torchvision module to extract and transfrom data.

In [3]:
# Define tranformer ny normalising the data to have mean of 0 and standard deviation of 1
transform = transforms.Compose([
    transforms.ToTensor(), # Convert PIL Image to tensor
    transforms.Normalize((0.0,), (1.0,)),   # Normalise data using standardisation
    transforms.Lambda(lambda x: x.flatten())    # Flatten the image
])

# Install and transform training data
train_dataset = datasets.FashionMNIST(
    root="./data/train",
    train=True,
    download=True,
    transform=transform
)

# Install and transfrom test data
test_dataset = datasets.FashionMNIST(
    root="./data/test",
    train=False,
    download=True,
    transform=transform
)

In [4]:
# Split the train data into train and validation sets
train_size = int((5/6)*len(train_dataset))
val_size = len(train_dataset) - train_size

train_set, val_set = random_split(train_dataset, [train_size, val_size])

# Load datasets into DataLoader
train_loader = DataLoader(train_set, batch_size=50, shuffle=True)
val_loader = DataLoader(val_set, batch_size=50)
test_loader = DataLoader(test_dataset, batch_size=50)

In [5]:
# Check the size of the datasets
print(f"Training data size = {len(train_loader.dataset)}")
print(f"Validation data size = {len(val_loader.dataset)}")
print(f"Test data size = {len(test_loader.dataset)}")

data_batch, labels_batch = next(iter(train_loader))
print(f"Shape of the data batch: {data_batch.shape}")
print(f"Shape of the labels batch: {labels_batch.shape}")

Training data size = 50000
Validation data size = 10000
Test data size = 10000
Shape of the data batch: torch.Size([50, 784])
Shape of the labels batch: torch.Size([50])


# 2. Building and training a Baseline model

In [6]:
class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_dim: int, h1_dim: int, h2_dim: int, num_classes: int):
        super(MultiLayerPerceptron, self).__init__()
        # Defining layers of the MultiLayer perceptron
        self.hidden_layer1 = nn.Linear(input_dim, h1_dim)
        self.hidden_layer2 = nn.Linear(h1_dim, h2_dim)
        self.output_layer = nn.Linear(h2_dim, num_classes)

    def forward(self, mini_batch: t.Tensor) -> t.Tensor:
        # Pass input through the first hidden dimension and apply ReLu
        h1_out = t.relu(self.hidden_layer1(mini_batch))
        # Pass input through the second hidden dimension and apply ReLu
        h2_out = t.relu(self.hidden_layer2(h1_out))
        # Pass the result through the output layer and apply the sigmoid function
        props = t.sigmoid(self.output_layer(h2_out))
        return props
    
    def predict(self, mini_batch: t.Tensor) -> t.Tensor:
        logits = self.forward(mini_batch)
        return t.argmax(logits, dim=1)

In [12]:
def set_deterministic_seed(seed: int = 42) -> None:
    # Set all seeds for deterministic behavior
    t.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    # For CUDA machines that have CUDA
    if t.cuda.is_available():
        t.cuda.manual_seed(seed)
        t.cuda.manual_seed_all(seed)
        t.backends.cudnn.deterministic = True
        t.backends.cudnn.benchmark = False
        
def train_model(
        model: MultiLayerPerceptron,
        train_loader: DataLoader, 
        val_loader: DataLoader,
        criterion: nn,
        optimiser: t.optim, 
        num_epochs: int = 50, 
        lr: float=0.001,
        early_stopping: bool = False, 
        val_thrs: float = 0.85, 
        patience: int = 5
    ) -> dict:
    # Set seed for determinism
    set_deterministic_seed()

    # Track losses and accuracies
    training_loss, validation_loss = [], []
    training_accuracy, validation_accuracy = [], []

    epochs_no_improve = 0

    for _ in range(num_epochs):
        model.train()
        total_train_loss, correct_train, total_train_samples = 0.0, 0, 0
        # Iterate through the training set batch by batch
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            ce_loss = criterion(outputs, batch_y)

            # Reset the gradients from the previous steps
            optimiser.zero_grad()

            # Backpropagation
            ce_loss.backward()
            optimiser.step()

            # Update total training loss
            total_train_loss += ce_loss.item()

            # Compute accuracy efficiently
            with t.no_grad():
                train_preds = model.predict(batch_X)
                correct_train += (train_preds == batch_y).sum().item()
                total_train_samples += batch_y.size(0)

        avg_train_loss = total_train_loss / len(train_loader)
        train_acc = correct_train / total_train_samples

        # Compute validation loss and accuracy
        total_val_loss, correct_val, total_val_samples = 0.0, 0, 0
        model.eval()
        with t.no_grad():
            for batch_X_val, batch_y_val in val_loader:
                # Forward pass
                outputs_val = model(batch_X_val)
                val_loss = criterion(outputs_val, batch_y_val)
                total_val_loss += val_loss.item()

                # Compute accuracy efficiently
                with t.no_grad():
                    val_preds = model.predict(batch_X_val)
                    correct_val += (val_preds == batch_y_val).sum().item()
                    total_val_samples += batch_y_val.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = correct_val / total_val_samples

        # Save the metrics
        training_loss.append(round(avg_train_loss, 4))
        validation_loss.append(round(avg_val_loss, 4))
        training_accuracy.append(round(train_acc, 4))
        validation_accuracy.append(round(val_acc, 4))

        if early_stopping:
            if val_thrs <= val_acc:
                break

            if len(validation_accuracy) > 1 and validation_accuracy[-1] < validation_accuracy[-2]:
                epochs_no_improve += 1
            else:
                epochs_no_improve = 0

            if epochs_no_improve >= patience:
                break

    return {
        "Model": model,
        "Training Loss": training_loss,
        "Validation Loss": validation_loss,
        "Training Accuracy": training_accuracy,
        "Validation Accuracy": validation_accuracy
    }

def get_final_metrics(trained_model: MultiLayerPerceptron, train_loader: DataLoader, val_loader: DataLoader) -> dict:
    # Collect data and labels
    train_features, val_features = [], []
    train_labels, val_labels = [], []

    # Iterate over data loaders
    for t_feats, t_labels in train_loader:
        train_features.append(t_feats)
        train_labels.append(t_labels)

    for v_feats, v_labels in val_loader:
        val_features.append(v_feats)
        val_labels.append(v_labels)

    # Concatenate the Batches
    X_train = t.cat(train_features, dim=0)
    X_val = t.cat(val_features, dim=0)
    y_train = t.cat(train_labels, dim=0)
    y_val = t.cat(val_labels, dim=0)

    # Compute accuracies
    trained_model.eval()
    with t.no_grad():
        # Final training accuracy
        final_train_preds = trained_model.predict(X_train)
        final_train_acc = (final_train_preds == y_train).float().mean().item()

        # Final validation accuracy
        final_val_preds = trained_model.predict(X_val)
        final_val_acc = (final_val_preds == y_val).float().mean().item()

    return {
        "Final Train Acc": final_train_acc,
        "Final Val Acc": final_val_acc
    }

def plot_training_results(training_loss: list, validation_loss: list, training_accuracy: list, validation_accuracy: list) -> None:
    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(8, 6))
    # Left plot
    ax[0].plot(training_loss, marker='o', linestyle='--', color='blue', label='Train Loss')
    ax[0].plot(validation_loss, marker='o', linestyle='--', color='green', label="Val loss")
    ax[0].set_xlabel('Number of epochs')
    ax[0].set_ylabel('Loss (cross-entropy loss)')
    ax[0].set_title("Model Loss over Epochs")
    ax[0].grid(True)
    ax[0].legend()

    # Right plot
    ax[1].plot(training_accuracy, marker='o', linestyle='--', color='red', label='Train Acc')
    ax[1].plot(validation_accuracy, marker='o', linestyle='--', color='orange', label="Val Acc")
    ax[1].set_xlabel('Number of epochs')
    ax[1].set_ylabel('Accuracy (as a percentange)')
    ax[1].set_title("Model Accuracy over Epochs")
    ax[1].grid(True)
    ax[1].legend()
  
    fig.tight_layout()
    plt.show()

In [ ]:
"""Training my BaseLIne model"""
# Create an instance of the MultilayeredPerceptron
model = MultiLayerPerceptron(data_batch.shape[1], 128, 64, 10)
# Define a loss function and an optimiser using lr=0.001
criterion = nn.CrossEntropyLoss()
optimiser = t.optim.Adam(model.parameters(), lr=0.001)

# Train the model
train_results = train_model(model, train_loader, val_loader, criterion, optimiser, num_epochs=10)

final_metrics = get_final_metrics(train_results["Model"], train_loader, val_loader)

# Display baseline metrics
print(f"Baseline training accuracy = {final_metrics["Final Train Acc"]:.4f}")
print(f"Baseline validation accuracy = {final_metrics["Final Val Acc"]:.4f}")


In [9]:
# Display the training results
print(train_results["Training Loss"])

[1.6736654411554337, 1.6159108177423478, 1.5953989601135254, 1.5843462263345718, 1.5785300251245498, 1.5753557060956955, 1.5704507471323013, 1.5682201325893401, 1.553069157719612, 1.5408141534328461]
